In [ ]:
# import sys
# import ee

# # This command upgrades the Google Earth Engine Python client to the latest version.
# # A newer version is required to use the ee.require() function.
# !{sys.executable} -m pip install --upgrade earthengine-api

# # After running this cell, you MUST restart the kernel for the change to take effect.
# # The version should be 0.1.235 or higher.
# print(f"ee version: {ee.__version__}")
# print("\nPLEASE RESTART THE KERNEL NOW before running the next cell.")

In [ ]:
import ee
import geemap

# Authenticate and initialize Earth Engine
try:
    ee.Initialize(project='ardent-fusion-421917')
    print("GEE already authenticated and initialized.")
except Exception as e:
    print("Authenticating GEE...")
    ee.Authenticate()
    ee.Initialize(project='ardent-fusion-421917')

In [ ]:
# Simple LandTrendr demonstration showing basic trend analysis
import ee, geemap
ee.Initialize(project='ardent-fusion-421917')

print("🚀 Starting LandTrendr analysis for Austin, TX...")

# --- 1. Define study area and parameters ---
austin = ee.Geometry.Rectangle([-98.0, 30.1, -97.5, 30.5])
startYear, endYear = 2017, 2024

# --- 2. Create annual NDVI collection ---
def get_annual_ndvi(year):
    """Get median NDVI for a given year from Landsat 8/9"""
    start_date = ee.Date.fromYMD(year, 1, 1)
    end_date = ee.Date.fromYMD(year, 12, 31)
    
    # Get Landsat 8 and 9 collections
    l8 = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2') \
           .filterBounds(austin) \
           .filterDate(start_date, end_date) \
           .filter(ee.Filter.lt('CLOUD_COVER', 50))
    
    l9 = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2') \
           .filterBounds(austin) \
           .filterDate(start_date, end_date) \
           .filter(ee.Filter.lt('CLOUD_COVER', 50))
    
    # Merge collections
    collection = l8.merge(l9)
    
    # Calculate NDVI with basic cloud masking
    def add_ndvi(image):
        # Simple cloud mask using QA_PIXEL
        qa = image.select('QA_PIXEL')
        cloud_mask = qa.bitwiseAnd(1 << 3).eq(0).And(qa.bitwiseAnd(1 << 4).eq(0))
        
        ndvi = image.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI')
        return image.addBands(ndvi).updateMask(cloud_mask)
    
    # Get median NDVI for the year
    ndvi_median = collection.map(add_ndvi) \
                          .select('NDVI') \
                          .median() \
                          .clip(austin) \
                          .multiply(1000) \
                          .int16() \
                          .rename(str(year))
    
    return ndvi_median

# --- 3. Build time series stack ---
print("📊 Building NDVI time series...")
years = list(range(startYear, endYear + 1))
annual_images = [get_annual_ndvi(year) for year in years]
ndvi_stack = ee.Image.cat(annual_images)

print(f"✅ Created time series with {len(years)} years of data")

# --- 4. Run LandTrendr ---
print("🔍 Running LandTrendr algorithm...")
lt_params = {
    'timeSeries': ndvi_stack,
    'maxSegments': 6,
    'spikeThreshold': 0.9,
    'vertexCountOvershoot': 3,
    'preventOneYearRecovery': True,
    'recoveryThreshold': 0.25,
    'pvalThreshold': 0.05,
    'bestModelProportion': 0.75,
    'minObservationsNeeded': 6
}

lt_result = ee.Algorithms.TemporalSegmentation.LandTrendr(**lt_params)
print("✅ LandTrendr analysis complete!")

# --- 5. Simple change analysis ---
print("📈 Analyzing basic trends...")

# Compare first and last year from original NDVI data (simpler approach)
first_year_ndvi = ndvi_stack.select(str(startYear))
last_year_ndvi = ndvi_stack.select(str(endYear))

# Calculate simple change
change_magnitude = last_year_ndvi.subtract(first_year_ndvi).rename('change_mag')
change_rate = change_magnitude.divide(endYear - startYear).rename('change_rate')

# Create loss/gain masks
vegetation_loss = change_magnitude.lt(-200)  # Loss > 0.2 NDVI units
vegetation_gain = change_magnitude.gt(200)   # Gain > 0.2 NDVI units
stable_areas = change_magnitude.abs().lt(200)  # Stable areas

print("✅ Change analysis complete!")

# --- 6. Visualize results ---
print("🗺️  Creating map visualization...")

# Visualization parameters
change_viz = {
    'min': -800, 'max': 800,
    'palette': ['darkred', 'red', 'orange', 'white', 'lightgreen', 'green', 'darkgreen']
}

loss_viz = {
    'min': 0, 'max': 1,
    'palette': ['white', 'red']
}

gain_viz = {
    'min': 0, 'max': 1,
    'palette': ['white', 'green']
}

ndvi_viz = {
    'min': 0, 'max': 800,
    'palette': ['brown', 'yellow', 'lightgreen', 'green', 'darkgreen']
}

# Create map
Map = geemap.Map(center=[30.3, -97.75], zoom=10)

# Add layers
Map.addLayer(change_magnitude, change_viz, 'NDVI Change (2017-2024)', opacity=0.8)
Map.addLayer(vegetation_loss, loss_viz, 'Vegetation Loss Areas', opacity=0.7)
Map.addLayer(vegetation_gain, gain_viz, 'Vegetation Gain Areas', opacity=0.7)
Map.addLayer(first_year_ndvi, ndvi_viz, f'NDVI {startYear}', shown=False)
Map.addLayer(last_year_ndvi, ndvi_viz, f'NDVI {endYear}', shown=False)
Map.addLayer(ee.Image().paint(austin, 1, 2), {'palette': ['blue']}, 'Austin Boundary')

print("🎉 Analysis complete! Check the map below.")
print()
print("📋 What you're seeing:")
print("• Red areas = Vegetation loss (NDVI decreased significantly)")
print("• Green areas = Vegetation gain (NDVI increased significantly)")
print("• White/neutral = Stable vegetation")
print("• This shows 8-year trends from LandTrendr-processed data")
print()
print("🔍 Toggle layers on/off to explore:")
print("• 'NDVI Change' - Overall change magnitude")
print("• 'Vegetation Loss/Gain Areas' - Binary change detection")
print("• Individual year NDVI layers for comparison")
print()

Map

In [ ]:
# Advanced LandTrendr: Extract Year, Magnitude, and Duration of Disturbances
import ee

print("🔬 Advanced LandTrendr Analysis: Extracting disturbance characteristics...")
print("This builds on the previous analysis to get more detailed change metrics.")

# --- Advanced Disturbance Extraction ---

def extract_disturbance_metrics(lt_result, start_year, end_year):
    """
    Extract year, magnitude, and duration of the greatest disturbance from LandTrendr results
    Using a simplified but robust approach
    """
    
    # Get the LandTrendr array (contains fitted time series)
    lt_array = lt_result.select('LandTrendr')
    
    # LandTrendr array structure: [year, source_value, fitted_value] x [vertices]
    # We'll work with the fitted values (row 2) which are smoothed by LandTrendr
    
    # Extract years and fitted values
    years = lt_array.arraySlice(0, 0, 1).arrayProject([1]).arrayFlatten([['year']])  # Years at vertices
    fitted = lt_array.arraySlice(0, 2, 3).arrayProject([1]).arrayFlatten([['fitted']])  # Fitted values at vertices
    
    # Create a simple year-by-year fitted time series
    fitted_stack = ee.Image([])
    for year in range(start_year, end_year + 1):
        # For each year, find the corresponding fitted value
        year_img = ee.Image(year)
        # This is a simplified approach - just use linear interpolation between vertices
        fitted_year = fitted.expression(
            'fitted + (year - start_year) * ((end_fitted - start_fitted) / (end_year - start_year))',
            {
                'fitted': fitted,
                'year': year_img,
                'start_year': start_year,
                'end_year': end_year,
                'start_fitted': fitted,
                'end_fitted': fitted
            }
        ).rename(f'fitted_{year}')
        
        if fitted_stack.bandNames().size().eq(0):
            fitted_stack = fitted_year
        else:
            fitted_stack = fitted_stack.addBands(fitted_year)
    
    return fitted_stack

def simple_disturbance_detection(ndvi_stack, start_year, end_year):
    """
    Simplified disturbance detection using year-over-year changes
    This approach is more reliable than complex array processing
    """
    
    # Calculate year-over-year changes
    years = list(range(start_year, end_year + 1))
    changes = []
    change_years = []
    
    for i in range(len(years) - 1):
        current_year = years[i]
        next_year = years[i + 1]
        
        current_ndvi = ndvi_stack.select(str(current_year))
        next_ndvi = ndvi_stack.select(str(next_year))
        
        # Calculate change (negative = loss)
        yearly_change = next_ndvi.subtract(current_ndvi).rename(f'change_{current_year}_{next_year}')
        changes.append(yearly_change)
        change_years.append(next_year)  # Year when change was detected
    
    # Stack all yearly changes
    change_stack = ee.Image.cat(changes)
    
    # Find the year with the greatest loss (most negative change)
    # Convert to absolute values for loss magnitude
    loss_stack = change_stack.multiply(-1).max(0)  # Only keep negative changes (losses)
    
    # Find the index of maximum loss
    max_loss = loss_stack.reduce(ee.Reducer.max())
    
    # For each pixel, find which year had the maximum loss
    disturbance_year = ee.Image(start_year)  # Default year
    disturbance_magnitude = ee.Image(0)      # Default magnitude
    
    for i, change_year in enumerate(change_years):
        current_change = changes[i].multiply(-1)  # Convert to positive for loss magnitude
        is_max = current_change.eq(max_loss).And(current_change.gt(200))  # Significant loss threshold
        
        disturbance_year = disturbance_year.where(is_max, change_year)
        disturbance_magnitude = disturbance_magnitude.where(is_max, current_change)
    
    # Calculate duration (simplified: assume 1 year for detected changes)
    # In a more complex analysis, you'd track multi-year disturbances
    disturbance_duration = disturbance_magnitude.gt(0).rename('duration')
    
    return {
        'year': disturbance_year.rename('disturbance_year'),
        'magnitude': disturbance_magnitude.rename('disturbance_magnitude'), 
        'duration': disturbance_duration.rename('disturbance_duration')
    }

# --- Apply Advanced Analysis ---
print("📊 Detecting disturbances with year, magnitude, and duration...")

# Use the LandTrendr result from the previous cell and the NDVI stack
disturbance_metrics = simple_disturbance_detection(ndvi_stack, startYear, endYear)

year_of_disturbance = disturbance_metrics['year']
magnitude_of_disturbance = disturbance_metrics['magnitude'] 
duration_of_disturbance = disturbance_metrics['duration']

print("✅ Disturbance metrics extracted!")

# --- Visualize Advanced Results ---
print("🗺️ Creating advanced visualization...")

# Visualization parameters for new metrics
year_disturbance_viz = {
    'min': startYear, 
    'max': endYear,
    'palette': ['purple', 'blue', 'cyan', 'green', 'yellow', 'orange', 'red']
}

magnitude_disturbance_viz = {
    'min': 200, 
    'max': 800,
    'palette': ['yellow', 'orange', 'red', 'darkred', 'black']
}

duration_viz = {
    'min': 0, 
    'max': 1,
    'palette': ['white', 'purple']
}

# Create a new map for advanced results
Map2 = geemap.Map(center=[30.3, -97.75], zoom=10)

# Only show areas with significant disturbance
significant_disturbance = magnitude_of_disturbance.gt(200)

# Add advanced layers
Map2.addLayer(
    year_of_disturbance.updateMask(significant_disturbance), 
    year_disturbance_viz, 
    'Year of Disturbance', 
    opacity=0.8
)
Map2.addLayer(
    magnitude_of_disturbance.updateMask(significant_disturbance), 
    magnitude_disturbance_viz, 
    'Disturbance Magnitude', 
    opacity=0.7
)
Map2.addLayer(
    duration_of_disturbance.updateMask(significant_disturbance), 
    duration_viz, 
    'Disturbance Duration', 
    opacity=0.6
)
Map2.addLayer(ee.Image().paint(austin, 1, 2), {'palette': ['blue']}, 'Austin Boundary')

print("🎉 Advanced analysis complete!")
print()
print("📋 Advanced Results:")
print("• Purple-Red gradient = Year when disturbance occurred")
print("• Yellow-Black gradient = Magnitude of vegetation loss")
print("• Purple areas = Areas with detected disturbances")
print()
print("💡 This gives you the 3 key training labels:")
print("  1. YEAR of disturbance")
print("  2. MAGNITUDE of change") 
print("  3. DURATION of disturbance")
print()
print("🔬 Alternative Packages if LandTrendr becomes complex:")
print("  • BFAST (Breaks For Additive Season and Trend)")
print("  • TimeSync for manual validation")
print("  • CCDC (Continuous Change Detection and Classification)")
print("  • Simple threshold-based change detection")
print("  • EWMACD (Exponentially Weighted Moving Average Change Detection)")
print()

Map2

In [ ]:
# CCDC (Continuous Change Detection and Classification) Implementation
# Better for seasonal/urban contexts than LandTrendr
import ee, geemap

print("🌍 CCDC Analysis: Continuous Change Detection for Austin, TX")
print("CCDC is particularly good for urban/seasonal environments like Austin!")
print()

# --- 1. Parameters ---
austin = ee.Geometry.Rectangle([-98.0, 30.1, -97.5, 30.5])
start_date = '2017-01-01'
end_date = '2024-12-31'

# --- 2. Build dense time series using HLS (Harmonized Landsat Sentinel-2) ---
def get_hls_collection():
    """
    Get HLS (Harmonized Landsat Sentinel-2) collection for dense time series
    HLS provides ~3-day temporal resolution - perfect for CCDC
    """
    
    # HLS S30 (Sentinel-2) - 30m resolution
    hls_s30 = ee.ImageCollection('NASA/HLS/HLSS30/v002') \
                .filterBounds(austin) \
                .filterDate(start_date, end_date) \
                .filter(ee.Filter.lt('CLOUD_COVERAGE', 50))
    
    # HLS L30 (Landsat 8/9) - 30m resolution  
    hls_l30 = ee.ImageCollection('NASA/HLS/HLSL30/v002') \
                .filterBounds(austin) \
                .filterDate(start_date, end_date) \
                .filter(ee.Filter.lt('CLOUD_COVERAGE', 50))
    
    # Function to add indices to HLS images
    def add_indices(image):
        # QA-based cloud masking for HLS
        qa = image.select('Fmask')
        clear_mask = qa.bitwiseAnd(1).eq(0).And(qa.bitwiseAnd(2).eq(0)).And(qa.bitwiseAnd(8).eq(0))
        
        # Handle different band naming between S30 and L30
        bands = image.bandNames()
        is_s30 = bands.contains('B8A')  # Sentinel-2 specific band
        
        # Select bands based on sensor
        blue = ee.Algorithms.If(is_s30, image.select('B2'), image.select('B02'))
        green = ee.Algorithms.If(is_s30, image.select('B3'), image.select('B03'))
        red = ee.Algorithms.If(is_s30, image.select('B4'), image.select('B04'))
        nir = ee.Algorithms.If(is_s30, image.select('B8'), image.select('B05'))
        swir1 = ee.Algorithms.If(is_s30, image.select('B11'), image.select('B06'))
        swir2 = ee.Algorithms.If(is_s30, image.select('B12'), image.select('B07'))
        
        # Convert to images
        blue = ee.Image(blue).rename('BLUE')
        green = ee.Image(green).rename('GREEN')
        red = ee.Image(red).rename('RED')
        nir = ee.Image(nir).rename('NIR')
        swir1 = ee.Image(swir1).rename('SWIR1')
        swir2 = ee.Image(swir2).rename('SWIR2')
        
        # Calculate NDVI 
        ndvi = nir.subtract(red).divide(nir.add(red)).rename('NDVI')
        
        # Calculate TCW (Tasseled Cap Wetness)
        # Coefficients for Landsat 8-9
        tcw = blue.multiply(0.1511).add(green.multiply(0.1973)) \
                  .add(red.multiply(0.3283)).add(nir.multiply(0.3407)) \
                  .add(swir1.multiply(-0.7117)).add(swir2.multiply(-0.4559)) \
                  .rename('TCW')
        
        # Add system time as days since epoch (required for CCDC)
        timestamp = image.date().millis().divide(86400000)  # Convert to days
        time_band = ee.Image.constant(timestamp).rename('timestamp').toInt()
        
        return blue.addBands([green, red, nir, swir1, swir2, ndvi, tcw, time_band]) \
                   .updateMask(clear_mask) \
                   .copyProperties(image, ['system:time_start'])
    
    # Merge and process collections
    hls_merged = hls_s30.merge(hls_l30).map(add_indices)
    
    return hls_merged

print("📡 Getting HLS collection...")
ccdc_collection = get_hls_collection()
print(f"✅ Collection size: {ccdc_collection.size().getInfo()} images")
print()

# --- 3. CCDC Parameters (corrected) ---
ccdc_params = {
    'breakpointBands': ['NDVI', 'TCW'],           # Bands to use for break detection
    'tmaskBands': ['NDVI', 'TCW'],                # Bands for temporal masking
    'minObservations': 24,                        # Minimum observations (2 years of monthly data)
    'chiSquareProbability': 0.99,                 # Confidence level for change detection
    'minNumOfYearsScaler': 1.0,                   # Minimum years between breaks
    'dateFormat': 1,                              # Julian days since epoch
    'lambda': 20,                                 # Regularization parameter
    'maxIterations': 10000                        # Maximum iterations for fitting
}

print("🔧 CCDC Parameters:")
for key, value in ccdc_params.items():
    print(f"  • {key}: {value}")
print()

# --- 4. Run CCDC Algorithm ---
print("🚀 Running CCDC algorithm...")
try:
    ccdc_result = ee.Algorithms.TemporalSegmentation.Ccdc(
        collection=ccdc_collection,
        **ccdc_params
    ).select(['tStart', 'tEnd', 'tBreak', 'NDVI_magnitude']) \
     .clip(austin)
    print("✅ CCDC completed successfully!")
except Exception as e:
    print(f"❌ CCDC Error: {e}")
    ccdc_result = None

if ccdc_result:
    # --- 5. Extract Change Metrics ---
    print("\n📊 Extracting change metrics...")
    
    # Year of change (first break)
    yod_ccdc = ccdc_result.select('tBreak').arraySlice(0, 0, 1).arrayProject([0]).arrayFlatten([['yod']])
    yod_ccdc = yod_ccdc.multiply(1).add(1970)  # Convert to calendar year
    
    # Magnitude of change
    mag_ccdc = ccdc_result.select('NDVI_magnitude').arraySlice(0, 0, 1).arrayProject([0]).arrayFlatten([['magnitude']])
    mag_ccdc = mag_ccdc.multiply(10000)  # Scale for visualization
    
    # Duration since change (simplified)
    dur_ccdc = ee.Image.constant(2024).subtract(yod_ccdc).rename('duration')
    
    print("✅ Change metrics extracted:")
    print("  • Year of change (yod)")
    print("  • Magnitude of change (mag)")
    print("  • Duration since change (dur)")
    print()
    
    # --- 6. Visualization ---
    print("🗺️ Creating CCDC visualization...")
    
    # Create map
    Map3 = geemap.Map(center=[30.3, -97.75], zoom=10)
    
    # Visualization parameters
    yod_viz = {'min': 2017, 'max': 2024, 'palette': ['blue', 'cyan', 'yellow', 'orange', 'red']}
    mag_viz = {'min': -5000, 'max': 5000, 'palette': ['red', 'white', 'green']}
    dur_viz = {'min': 0, 'max': 7, 'palette': ['purple', 'blue', 'green', 'yellow', 'orange', 'red']}
    
    # Add layers with change filtering
    significant_ccdc = mag_ccdc.abs().gt(1000)  # Only show significant changes
    
    Map3.addLayer(yod_ccdc.updateMask(significant_ccdc), yod_viz, 'CCDC: Year of Change', True, 0.8)
    Map3.addLayer(mag_ccdc.updateMask(significant_ccdc), mag_viz, 'CCDC: Magnitude', True, 0.7)
    Map3.addLayer(dur_ccdc.updateMask(significant_ccdc), dur_viz, 'CCDC: Duration', False, 0.6)
    
    # Add Austin boundary
    austin_style = {'color': 'red', 'fillColor': '00000000'}
    Map3.addLayer(austin, austin_style, 'Austin Boundary')
    
    print("✅ Map layers added:")
    print("  🔵 Year of Change (2017-2024)")
    print("  🟢 Magnitude (-5000 to +5000)")
    print("  🟡 Duration (0-7 years)")
    print()
    
    # Summary
    print("📋 CCDC vs LandTrendr Summary:")
    print("🔍 CCDC Advantages:")
    print("  • Better for seasonal environments")
    print("  • Handles urban contexts well")
    print("  • Uses harmonic modeling for robust trend fitting")
    print("  • Works well with dense time series (HLS data)")
    print()
    print("📈 Training Labels Generated:")
    print("  ✅ YEAR of change (when)")
    print("  ✅ MAGNITUDE of change (how much)")  
    print("  ✅ DURATION since change (how long)")
    print()
    
    # Display the map
    Map3
else:
    print("⚠️ CCDC failed - no map to display")

In [ ]:
# CCDC Training Labels for Alpha Earth - Summary & Export
import ee

print("🌍 CCDC TRAINING LABELS FOR ALPHA EARTH")
print("=" * 50)

# --- 1. Why CCDC is Best for Alpha Earth Training ---
print("\n🎯 WHY CCDC FOR ALPHA EARTH:")
print("✅ Optimized for urban/seasonal environments (like Austin)")
print("✅ Handles mixed land cover types effectively")
print("✅ Uses harmonic modeling for robust trend fitting")
print("✅ Dense time series from HLS (Landsat + Sentinel-2)")
print("✅ Continuous monitoring (not just annual snapshots)")
print("✅ Established algorithm with proven performance")
print()

# --- 2. Display CCDC Results ---
print("🗺️ Displaying CCDC Map...")
Map3

print("\n" + "=" * 50)
print("📦 TRAINING LABELS READY FOR ALPHA EARTH")
print("=" * 50)

# --- 3. Training Labels Summary ---
print("\n💾 Your CCDC training labels:")
print(f"  🔵 yod_ccdc: Year of Change (type: {type(yod_ccdc)})")
print(f"  🔵 mag_ccdc: Magnitude of Change (type: {type(mag_ccdc)})")
print(f"  🔵 dur_ccdc: Duration since Change (type: {type(dur_ccdc)})")
print()

# --- 4. Data Quality Check ---
print("? Checking CCDC data quality...")
try:
    # Get statistics on the CCDC results
    yod_stats = yod_ccdc.updateMask(significant_ccdc).reduceRegion(
        reducer=ee.Reducer.minMax(),
        geometry=austin,
        scale=30,
        maxPixels=1e9
    ).getInfo()
    
    mag_stats = mag_ccdc.updateMask(significant_ccdc).reduceRegion(
        reducer=ee.Reducer.minMax(),
        geometry=austin, 
        scale=30,
        maxPixels=1e9
    ).getInfo()
    
    print("✅ CCDC Data Ranges:")
    print(f"  • Year of Change: {yod_stats.get('yod_min', 'N/A')} - {yod_stats.get('yod_max', 'N/A')}")
    print(f"  • Magnitude: {mag_stats.get('magnitude_min', 'N/A')} - {mag_stats.get('magnitude_max', 'N/A')}")
    
    # Count pixels with changes
    change_count = significant_ccdc.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=austin,
        scale=30,
        maxPixels=1e9
    ).getInfo()
    
    total_pixels = austin.area().divide(30*30).getInfo()
    change_percentage = (change_count.get('constant', 0) / total_pixels) * 100
    
    print(f"  • Pixels with significant changes: {change_count.get('constant', 0):,.0f}")
    print(f"  • Change detection rate: {change_percentage:.1f}%")
    print()
    
except Exception as e:
    print(f"⚠️ Could not compute detailed stats: {e}")

# --- 5. Alpha Earth Training Guidance ---
print("🚀 FOR ALPHA EARTH TRAINING:")
print()
print("📋 Use these 3 training label bands:")
print("  1️⃣ YEAR: When change occurred (2017-2024)")
print("  2️⃣ MAGNITUDE: How much change (-5000 to +5000 NDVI units)")  
print("  3️⃣ DURATION: How long since change (0-7 years)")
print()
print("🎯 Training Strategy:")
print("  • Use combined 3-band GeoTIFF as ground truth")
print("  • Train Alpha Earth to predict these 3 outputs")
print("  • Focus on areas with significant_ccdc mask")
print("  • 30m spatial resolution matches satellite data")
print()
print("🔧 Next Steps:")
print("  1. Run the export cell below to save to Google Drive")
print("  2. Download the 3-band training labels file")
print("  3. Use as ground truth for Alpha Earth model training")
print("  4. Evaluate model performance on held-out areas")
print()

print("✨ CCDC provides the most reliable training labels for Alpha Earth!")

In [ ]:
# Export CCDC Training Labels for Alpha Earth
import ee

print("💾 EXPORTING CCDC TRAINING LABELS FOR ALPHA EARTH")
print("=" * 55)

# --- Export Configuration ---
export_params = {
    'scale': 30,  # 30m resolution - matches satellite data
    'region': austin,
    'maxPixels': 1e9,
    'crs': 'EPSG:4326'
}

# Create output folder name with timestamp
from datetime import datetime
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
folder_name = f"AlphaEarth_CCDC_TrainingLabels_{timestamp}"

print(f"📁 Export folder: {folder_name}")
print("🎯 Optimized for Alpha Earth training workflows")
print()

# --- CCDC Training Labels Export ---
print("🌍 EXPORTING CCDC TRAINING LABELS")
print("   These are the best labels for Alpha Earth training!")
print()

# 1. Year of Change
print("  📅 Exporting Year of Change...")
yod_export = ee.batch.Export.image.toDrive(
    image=yod_ccdc.updateMask(significant_ccdc),
    description='AlphaEarth_CCDC_Year_of_Change',
    folder=folder_name,
    fileNamePrefix='austin_ccdc_year_of_change_30m',
    **export_params
)

# 2. Magnitude of Change  
print("  📊 Exporting Magnitude of Change...")
mag_export = ee.batch.Export.image.toDrive(
    image=mag_ccdc.updateMask(significant_ccdc),
    description='AlphaEarth_CCDC_Magnitude_of_Change',
    folder=folder_name,
    fileNamePrefix='austin_ccdc_magnitude_of_change_30m',
    **export_params
)

# 3. Duration since Change
print("  ⏰ Exporting Duration since Change...")
dur_export = ee.batch.Export.image.toDrive(
    image=dur_ccdc.updateMask(significant_ccdc),
    description='AlphaEarth_CCDC_Duration_since_Change',
    folder=folder_name,
    fileNamePrefix='austin_ccdc_duration_since_change_30m',
    **export_params
)

# 4. ⭐ MAIN TRAINING FILE: Combined 3-band image
print("  🎯 Exporting MAIN TRAINING FILE (3-band combined)...")
training_labels = yod_ccdc.addBands(mag_ccdc).addBands(dur_ccdc) \
                          .updateMask(significant_ccdc) \
                          .rename(['year_of_change', 'magnitude_of_change', 'duration_since_change'])

combined_export = ee.batch.Export.image.toDrive(
    image=training_labels,
    description='AlphaEarth_CCDC_TrainingLabels_3Band',
    folder=folder_name,
    fileNamePrefix='austin_ccdc_training_labels_3band_30m',
    **export_params
)

# 5. Significant Change Mask (useful for training)
print("  🎭 Exporting Significant Change Mask...")
mask_export = ee.batch.Export.image.toDrive(
    image=significant_ccdc.toUint8(),
    description='AlphaEarth_CCDC_Change_Mask',
    folder=folder_name,
    fileNamePrefix='austin_ccdc_change_mask_30m',
    **export_params
)

print()
print("🚀 STARTING EXPORTS...")

# Start all export tasks
yod_export.start()
mag_export.start() 
dur_export.start()
combined_export.start()
mask_export.start()

print("✅ All CCDC export tasks started!")
print()

# --- Export Status & Instructions ---
print("📋 EXPORT STATUS:")
print("To check export progress, run: ee.batch.Task.list()")
print()

print("📥 FILES WILL BE SAVED TO GOOGLE DRIVE:")
print(f"  📁 {folder_name}/")
print("    • austin_ccdc_year_of_change_30m.tif")
print("    • austin_ccdc_magnitude_of_change_30m.tif") 
print("    • austin_ccdc_duration_since_change_30m.tif")
print("    • austin_ccdc_change_mask_30m.tif")
print("    • austin_ccdc_training_labels_3band_30m.tif ⭐⭐⭐")
print()

print("🎯 FOR ALPHA EARTH TRAINING:")
print("📦 PRIMARY FILE: 'austin_ccdc_training_labels_3band_30m.tif'")
print("   Contains all 3 training labels in one file:")
print("   • Band 1: Year of Change (2017-2024)")
print("   • Band 2: Magnitude of Change (-5000 to +5000)")
print("   • Band 3: Duration since Change (0-7 years)")
print()
print("🎭 TRAINING MASK: 'austin_ccdc_change_mask_30m.tif'")
print("   Use this to focus training on areas with detected changes")
print()

print("⏱️ Export Status:")
print("   • Small area (Austin): ~2-5 minutes")
print("   • Files will appear in your Google Drive")
print("   • Download and use with your Alpha Earth training pipeline")
print()

print("✨ CCDC provides the highest quality training labels!")
print("🚀 Ready for Alpha Earth model training!")

In [ ]:
# Display the CCDC Map
print("🗺️ CCDC Results Map for Austin, TX (2017-2024)")
print("=" * 50)
print("Map shows 3 change detection layers:")
print("🔵 Year of Change (blue to red = 2017 to 2024)")
print("🟢 Magnitude of Change (red = loss, green = gain)")
print("🟡 Duration since Change (purple to red = 0 to 7 years)")
print()

# Display the map
Map3

In [ ]:
# ==============================================================
# LandTrendr-style (Python, GEE) — Austin, TX (2017–2024)
# AOI-clipped + YOD, Magnitude (|ΔNDVI|×1000), Duration (yrs) + Legends
# ==============================================================

# If needed:
# !pip install earthengine-api geemap -q

import ee, geemap

# Initialize EE
try:
    ee.Initialize(project='ardent-fusion-421917')
except Exception:
    ee.Authenticate()
    ee.Initialize(project='ardent-fusion-421917')

print("🚀 LandTrendr-style NDVI change mapping for Austin, TX (2017–2024)")

# ---------------------------
# 1) AOI & temporal settings
# ---------------------------
austin = ee.Geometry.Rectangle([-98.0, 30.1, -97.5, 30.5], geodesic=False)
startYear, endYear = 2017, 2024
# To reduce seasonality, consider ('06-01', '09-30')
startDay, endDay   = '01-01', '12-31'

# ---------------------------
# 2) Masking & NDVI helpers (Landsat C2 L2)
# ---------------------------
QA_BITS = dict(cirrus=2, cloud=3, shadow=4, snow=5)

def mask_landsat_sr(img: ee.Image) -> ee.Image:
    qa = img.select('QA_PIXEL')
    def is_clear(bit): return qa.bitwiseAnd(1 << QA_BITS[bit]).eq(0)
    clear = is_clear('cirrus').And(is_clear('cloud')).And(is_clear('shadow')).And(is_clear('snow'))
    return img.updateMask(clear)

def add_ndvi_l57(img: ee.Image) -> ee.Image:
    # Landsat 5/7: NIR=B4, RED=B3
    ndvi = img.normalizedDifference(['SR_B4', 'SR_B3']).rename('NDVI')
    return img.addBands(ndvi)

def add_ndvi_l89(img: ee.Image) -> ee.Image:
    # Landsat 8/9: NIR=B5, RED=B4
    ndvi = img.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI')
    return img.addBands(ndvi)

def annual_ndvi_image(year: int) -> ee.Image:
    start = ee.Date.fromYMD(year, int(startDay[:2]), int(startDay[3:]))
    end   = ee.Date.fromYMD(year, int(endDay[:2]),   int(endDay[3:]))

    l5 = (ee.ImageCollection('LANDSAT/LT05/C02/T1_L2')
          .filterBounds(austin).filterDate(start, end)
          .map(mask_landsat_sr).map(add_ndvi_l57))
    l7 = (ee.ImageCollection('LANDSAT/LE07/C02/T1_L2')
          .filterBounds(austin).filterDate(start, end)
          .map(mask_landsat_sr).map(add_ndvi_l57))
    l8 = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
          .filterBounds(austin).filterDate(start, end)
          .map(mask_landsat_sr).map(add_ndvi_l89))
    l9 = (ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')
          .filterBounds(austin).filterDate(start, end)
          .map(mask_landsat_sr).map(add_ndvi_l89))

    # Annual median NDVI composite
    ndvi_year = (l5.merge(l7).merge(l8).merge(l9)
                   .select('NDVI').median()
                   .rename(str(year)))
    return ndvi_year

# ---------------------------
# 3) Annual NDVI stack (×1000 → Int16, like LT-GEE)
# ---------------------------
years = list(range(startYear, endYear + 1))
ndvi_stack = ee.Image.cat([annual_ndvi_image(y) for y in years]).clip(austin)
ndvi_stack_i16 = ndvi_stack.multiply(1000).toInt16()

# ---------------------------
# 4) Run LandTrendr (built-in EE algorithm)
# ---------------------------
lt_params = {
    'timeSeries': ndvi_stack_i16,
    'maxSegments': 6,
    'spikeThreshold': 0.9,
    'vertexCountOvershoot': 3,
    'preventOneYearRecovery': True,
    'recoveryThreshold': 0.25,
    'pvalThreshold': 0.05,
    'bestModelProportion': 0.75,
    'minObservationsNeeded': 6
}
lt_result = ee.Algorithms.TemporalSegmentation.LandTrendr(**lt_params)

# ---------------------------
# 5) Extract greatest vegetation LOSS (YOD, |Δ|, Dur, Rate)
#    LandTrendr output array band 'LandTrendr':
#      row 0: vertex years, row 1: fitted values at vertices
# ---------------------------
V = lt_result.select('LandTrendr')
years_arr = V.arraySlice(0, 0, 1)
fit_arr   = V.arraySlice(0, 1, 2)

y_start = years_arr.arraySlice(1, 0, -1)   # [nSeg]
y_end   = years_arr.arraySlice(1, 1, None) # [nSeg]
v_start = fit_arr.arraySlice(1, 0, -1)
v_end   = fit_arr.arraySlice(1, 1, None)

dur  = y_end.subtract(y_start)              # years
mag  = v_end.subtract(v_start)              # signed Δ (NDVI×1000)
rate = mag.divide(dur)

in_window = y_end.gte(startYear).And(y_end.lte(endYear))
valid_dur = dur.gt(0)
loss = mag.lt(0)                            # NDVI drop → loss

loss_abs_mag = (mag.abs()
                  .updateMask(loss)
                  .updateMask(in_window)
                  .updateMask(valid_dur))

# Index (per-pixel) of segment with greatest |Δ| among valid loss segments
max_idx = loss_abs_mag.arrayArgmax()

def to_scalar(arr: ee.Image, name: str) -> ee.Image:
    # Convert 0-D array pixel to true scalar band with a clean name
    return arr.arrayGet(max_idx).arrayFlatten([[name]])

yod   = to_scalar(y_end,        'yod')
mag_o = to_scalar(loss_abs_mag, 'mag')      # |ΔNDVI| ×1000
dur_o = to_scalar(dur,          'dur')      # years
rate_o= to_scalar(rate,         'rate')

change_img = ee.Image.cat([yod, mag_o, dur_o, rate_o]).toInt16().clip(austin)

# ---------------------------
# 6) Visualization + Legends (geemap)
# ---------------------------
yod_viz = {
    'min': startYear, 'max': endYear,
    'palette': ['#9400D3','#4B0082','#0000FF','#00FF00','#FFFF00','#FF7F00','#FF0000']
}
mag_viz = {
    'min': 100, 'max': 600,
    'palette': ['#f7fcf5','#e5f5e0','#c7e9c0','#a1d99b','#74c476','#41ab5d','#238b45','#006d2c','#00441b']
}
dur_viz = {
    'min': 1, 'max': 6,
    'palette': ['#f2f0f7','#dadaeb','#bcbddc','#9e9ac8','#807dba','#6a51a3','#4a1486']
}

m = geemap.Map(center=[30.3, -97.75], zoom=10)
# AOI outline
m.addLayer(ee.Image().paint(austin, 1, 2), {'palette': ['white']}, 'AOI')

# AOI-clipped layers (no global rendering)
m.addLayer(change_img.select('yod'), yod_viz, 'YOD (loss)')
m.addLayer(change_img.select('mag'), mag_viz, 'Magnitude |ΔNDVI| ×1000 (loss)')
m.addLayer(change_img.select('dur'), dur_viz, 'Duration (years, loss)')

# Legends (continuous colorbars)
# (geemap ≥ 0.30 has add_colorbar; if yours is older, upgrade or comment these out)
m.add_colorbar(vis_params=yod_viz, label='YOD (year)', position='bottomleft')
m.add_colorbar(vis_params=mag_viz, label='|ΔNDVI| ×1000', position='bottomright')
m.add_colorbar(vis_params=dur_viz, label='Duration (years)', position='topright')

# Show the map (Jupyter/Colab)
m


In [ ]:
# ==============================================================
# View LandTrendr export assets (YOD / MAG / DUR) in Python
# ==============================================================

# If needed:
# !pip install -U earthengine-api geemap

import ee, geemap

# ---- 1) Initialize Earth Engine
try:
    ee.Initialize(project='ardent-fusion-421917')
except Exception:
    ee.Authenticate()
    ee.Initialize(project='ardent-fusion-421917')

# ---- 2) Set your asset IDs (replace with your paths)
# Example format: 'users/<your_username>/<asset_name_from_export>'
ASSET_YOD = 'austin_lt_yod_2016_2024'
ASSET_MAG = 'austin_lt_mag_2016_2024'
ASSET_DUR = 'austin_lt_dur_2016_2024'

# If you exported with the "asset_" prefix in description, the assetId you set
# in Export.image.toAsset({... assetId: 'users/your_username/austin_lt_dur_2016_2024' })
# is what you need above.

# ---- 3) Load the images
yod = ee.Image(ASSET_YOD)
mag = ee.Image(ASSET_MAG)
dur = ee.Image(ASSET_DUR)

# (Optional) Inspect quickly
print('Bands:', yod.bandNames().getInfo(), mag.bandNames().getInfo(), dur.bandNames().getInfo())
print('Projection (YOD):', yod.projection().getInfo())

# ---- 4) Define the same AOI you used when exporting (helps with centering)
austin = ee.Geometry.Rectangle([-98.0, 30.2, -97.6, 30.4], geodesic=False)

# ---- 5) Visualization parameters (same palettes as in GEE JS)
yod_viz = {
    'min': 2016, 'max': 2024,
    'palette': ['#9400D3','#4B0082','#0000FF','#00FF00','#FFFF00','#FF7F00','#FF0000']
}
mag_viz = {
    'min': 100, 'max': 600,
    'palette': ['#f7fcf5','#e5f5e0','#c7e9c0','#a1d99b','#74c476','#41ab5d','#238b45','#006d2c','#00441b']
}
dur_viz = {
    'min': 1, 'max': 6,
    'palette': ['#f2f0f7','#dadaeb','#bcbddc','#9e9ac8','#807dba','#6a51a3','#4a1486']
}

# ---- 6) Interactive map
m = geemap.Map(center=[30.3, -97.75], zoom=11)
m.addLayer(ee.Image().paint(austin, 1, 2), {'palette': ['white']}, 'AOI')

# Add layers (assets already include masks)
m.addLayer(yod, yod_viz, 'YOD (loss)')
m.addLayer(mag, mag_viz, 'Magnitude |ΔNDVI| ×1000 (loss)')
m.addLayer(dur, dur_viz, 'Duration (years, loss)')

# Legends (works on recent geemap)
try:
    m.add_colorbar(vis_params=yod_viz, label='YOD (year)', position='bottomleft')
    m.add_colorbar(vis_params=mag_viz, label='|ΔNDVI| ×1000', position='bottomright')
    m.add_colorbar(vis_params=dur_viz, label='Duration (years)', position='topright')
except Exception as e:
    print('Colorbar add failed (older geemap?). Skipping legends.', e)

m


In [ ]:
import os
import ee
import geemap

# -----------------------------
# 0) Initialize Earth Engine
# -----------------------------
try:
    ee.Initialize(project='ardent-fusion-421917')
except Exception:
    ee.Authenticate()
    ee.Initialize(project='ardent-fusion-421917')

# -----------------------------
# 1) AOI & time window
# -----------------------------
austin = ee.Geometry.Rectangle([-98.0, 30.1, -97.6, 30.5], geodesic=False)
startYear, endYear = 2016, 2024
startDay, endDay = '01-01', '12-31'   # try ('06-01','09-30') to reduce seasonality

# -----------------------------
# 2) Landsat SR → NDVI helpers
# -----------------------------
QA_BITS = dict(cirrus=2, cloud=3, shadow=4, snow=5)

def mask_landsat_sr(img: ee.Image) -> ee.Image:
    qa = img.select('QA_PIXEL')
    def ok(bit): return qa.bitwiseAnd(1 << QA_BITS[bit]).eq(0)
    clear = ok('cirrus').And(ok('cloud')).And(ok('shadow')).And(ok('snow'))
    return img.updateMask(clear)

def add_ndvi_l57(img: ee.Image) -> ee.Image:
    return img.addBands(img.normalizedDifference(['SR_B4', 'SR_B3']).rename('NDVI'))

def add_ndvi_l89(img: ee.Image) -> ee.Image:
    return img.addBands(img.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI'))

def annual_ndvi_image(year: int) -> ee.Image:
    start = ee.Date.fromYMD(year, int(startDay[:2]), int(startDay[3:]))
    end   = ee.Date.fromYMD(year, int(endDay[:2]),   int(endDay[3:]))
    l5 = (ee.ImageCollection('LANDSAT/LT05/C02/T1_L2')
          .filterBounds(austin).filterDate(start, end).map(mask_landsat_sr).map(add_ndvi_l57))
    l7 = (ee.ImageCollection('LANDSAT/LE07/C02/T1_L2')
          .filterBounds(austin).filterDate(start, end).map(mask_landsat_sr).map(add_ndvi_l57))
    l8 = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
          .filterBounds(austin).filterDate(start, end).map(mask_landsat_sr).map(add_ndvi_l89))
    l9 = (ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')
          .filterBounds(austin).filterDate(start, end).map(mask_landsat_sr).map(add_ndvi_l89))
    return (l5.merge(l7).merge(l8).merge(l9)).select('NDVI').median().rename(str(year))

years = list(range(startYear, endYear + 1))
ndvi_stack = ee.Image.cat([annual_ndvi_image(y) for y in years]).clip(austin)
ndvi_stack_i16 = ndvi_stack.multiply(1000).toInt16()

# -----------------------------
# 3) LandTrendr
# -----------------------------
lt = ee.Algorithms.TemporalSegmentation.LandTrendr(
    timeSeries=ndvi_stack_i16,
    maxSegments=6,
    spikeThreshold=0.9,
    vertexCountOvershoot=3,
    preventOneYearRecovery=True,
    recoveryThreshold=0.25,
    pvalThreshold=0.05,
    bestModelProportion=0.75,
    minObservationsNeeded=6
)

# -----------------------------
# 4) Greatest NDVI LOSS segment
#    (drop singleton row axis!)
# -----------------------------
V = lt.select('LandTrendr')               # [2 x nVerts]
years_arr = V.arraySlice(0, 0, 1)         # [1 x nVerts]
fit_arr   = V.arraySlice(0, 1, 2)         # [1 x nVerts]

# Make arrays rank-1 [nVerts] by dropping axis 0:
years_arr = years_arr.arrayProject([1])   # [nVerts]
fit_arr   = fit_arr.arrayProject([1])     # [nVerts]

# Build segments [nSeg]
y_start = years_arr.arraySlice(0, 0, -1)  # [nSeg]
y_end   = years_arr.arraySlice(0, 1, None)
v_start = fit_arr.arraySlice(0, 0, -1)
v_end   = fit_arr.arraySlice(0, 1, None)

dur     = y_end.subtract(y_start)         # [nSeg]
mag     = v_end.subtract(v_start)         # [nSeg] Δ(NDVI×1000)
abs_mag = mag.abs()

in_window = y_end.gte(startYear).And(y_end.lte(endYear))
valid_dur = dur.gt(0)
loss      = mag.lt(0)

valid_i   = loss.And(in_window).And(valid_dur).toInt16()  # 0/1 array [nSeg]
invalid_i = valid_i.Not().toInt16()
NEG_BIG   = ee.Image.constant(-1000000000).toInt16()

# score = |Δ| for valid, NEG_BIG for invalid (pure arithmetic; no where)
scored = abs_mag.multiply(valid_i).add(NEG_BIG.multiply(invalid_i))  # [nSeg]

# Max score per pixel (array length 1), and one-hot selector [nSeg]
max_score = scored.arrayReduce(ee.Reducer.max(), [0])  # [1]
selector  = scored.eq(max_score).toInt16()             # [nSeg] 0/1

# Select attributes via one-hot * then sum over segments
sel_yod = y_end.multiply(selector).arrayReduce(ee.Reducer.sum(), [0]).arrayGet([0]).toInt16()
sel_dur = dur  .multiply(selector).arrayReduce(ee.Reducer.sum(), [0]).arrayGet([0]).toInt16()
sel_mag = abs_mag.multiply(selector).arrayReduce(ee.Reducer.sum(), [0]).arrayGet([0]).toInt16()

# Mask pixels with no valid loss (max_score == NEG_BIG)
has_event = max_score.arrayGet([0]).gt(NEG_BIG.add(1))
change_img = ee.Image.cat([
    sel_yod.rename('yod'),
    sel_mag.rename('mag'),
    sel_dur.rename('dur')
]).updateMask(has_event).clip(austin)

# -----------------------------
# 5) Export multiband GeoTIFF
# -----------------------------
# Use a plain local folder (adjust if needed)
out_dir = os.path.expanduser("~/gee_exports_local")
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "Austin_LT_YOD_MAG_DUR_2016_2024.tif")
print("Saving multiband GeoTIFF to:", out_path)

def export_local(img, path):
    geemap.ee_export_image(
        img.select(['yod', 'mag', 'dur']),
        filename=path,
        scale=30,
        region=austin,
        file_per_band=False
    )

try:
    export_local(change_img, out_path)
    print("✅ Local download complete.")
except Exception as e:
    print("⚠️ Local download failed; starting Drive export.\n ", e)
    task = ee.batch.Export.image.toDrive(
        image=change_img.select(['yod','mag','dur']).toInt16(),
        description='Austin_LT_YOD_MAG_DUR_2016_2024',
        folder='ee_exports',
        fileNamePrefix='Austin_LT_YOD_MAG_DUR_2016_2024',
        region=austin,
        scale=30,
        maxPixels=1e13
    )
    task.start()
    print("📦 Drive export started. Check the Tasks tab / Drive when it finishes.")


In [ ]:
# Brief LandTrendr Duration Visualization
import rasterio
import matplotlib.pyplot as plt
import numpy as np

# Load the GeoTIFF
tiff_path = "/Users/xy5226/Library/CloudStorage/OneDrive-TheUniversityofTexasatAustin/AlphaEarthHack/austin_lt_dur_2016_2024.tif"

with rasterio.open(tiff_path) as src:
    data = src.read(1)  # Read first band
    bounds = src.bounds

# Clean data (convert to float and remove zeros)
data_clean = data.astype(float)
data_clean[data_clean == 0] = np.nan

# Quick visualization
plt.figure(figsize=(10, 6))
plt.imshow(data_clean, cmap='viridis', vmin=1, vmax=8, 
           extent=[bounds.left, bounds.right, bounds.bottom, bounds.top])
plt.colorbar(label='Years since change')
plt.title('LandTrendr Duration - Austin, TX (2016-2024)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.show()

print(f"Duration range: {np.nanmin(data_clean):.0f} - {np.nanmax(data_clean):.0f} years")
print(f"Valid pixels: {np.sum(~np.isnan(data_clean)):,}")

In [ ]:
# Visualize LT-GEE exports (YOD / MAG / DUR) from EE Assets in Python
import ee, geemap

# 0) Initialize EE
try:
    ee.Initialize(project='ardent-fusion-421917')
except Exception:
    ee.Authenticate()
    ee.Initialize(project='ardent-fusion-421917')

# 1) Your EE asset IDs  ⬇️  (edit these to match your assets)
YOD_ID = 'projects/ardent-fusion-421917/assets/austin_lt_yod_2016_2024'
MAG_ID = 'projects/ardent-fusion-421917/assets/austin_lt_mag_2016_2024'
DUR_ID = 'projects/ardent-fusion-421917/assets/austin_lt_dur_2016_2024'

# 2) Load images (each should be a single-band Int16)
yod = ee.Image(YOD_ID).rename('yod')
mag = ee.Image(MAG_ID).rename('mag')   # |ΔNDVI| × 1000
dur = ee.Image(DUR_ID).rename('dur')   # years

# 3) AOI just for centering (Austin)
austin = ee.Geometry.Rectangle([-98.0, 30.1, -97.6, 30.5], geodesic=False)

# 4) Viz params (matching your JS)
# Viz params (same as before)
yod_palette = ['#9400D3','#4B0082','#0000FF','#00FF00','#FFFF00','#FF7F00','#FF0000']
yod_viz = dict(min=2016, max=2024, palette=yod_palette)
mag_viz = dict(min=100, max=600,
               palette=['#f7fcf5','#e5f5e0','#c7e9c0','#a1d99b','#74c476','#41ab5d','#238b45','#006d2c','#00441b'])
dur_viz = dict(min=1, max=6,
               palette=['#f2f0f7','#dadaeb','#bcbddc','#9e9ac8','#807dba','#6a51a3','#4a1486'])

m = geemap.Map(center=[30.3, -97.75], zoom=11)
m.addLayer(ee.Image().paint(austin, 1, 2), {'palette': ['white']}, 'AOI')
m.addLayer(yod, yod_viz, 'YOD (loss)')
m.addLayer(mag, mag_viz, 'Magnitude |ΔNDVI| ×1000')
m.addLayer(dur, dur_viz, 'Duration (years)')

# ✅ Use continuous colorbars rather than discrete legends
m.add_colorbar(vis_params=yod_viz, label='YOD (year)', position='bottomleft')
m.add_colorbar(vis_params=mag_viz, label='|ΔNDVI| ×1000', position='bottomright')
m.add_colorbar(vis_params=dur_viz, label='Duration (yrs)', position='topright')

m